In [ ]:
library(dplyr)
library(tidyverse)
library(data.table)
library(ggplot2)
library(RColorBrewer)

library(Seurat)

options(stringsAsFactors=F)
options(scipen=100)

In [ ]:
# ---- Set the following paths to match your environment ----
data_path  = "/path/to/data"            # directory for data
output_path  = "/path/to/output"            # directory for output
# -----------------------------------------------------------

project_id.data <- Read10X(data.dir = data_path) 
min_cell=ncol(project_id.data)/100
Seurat_ob <- CreateSeuratObject(counts = project_id.data, project = "project_id", min.cells = min_cell)
filtered_count_df=as.data.frame(Seurat_ob[["RNA"]]$counts)
out_f=output_path
write.table(data.frame(gene=rownames(filtered_count_df), filtered_count_df), paste0(out_f, "/filtered_count.txt") ,row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

#option: convert SYMBOL to ENSG
annotation_df=read.table(paste0(data_path, "/features.tsv.gz")
colnames(annotation_df)=c("ENSG", "gene", "GENE", "Expression")
count_df=read.table(paste0(data_path, "/filtered_count.txt"), header=1)

#option: convert SYMBOL to ENSG, filter to unique gene id
count_df2=left_join(count_df, annotation_df, by="gene")
unique_df=count_df2[!(duplicated(count_df2$ENSG) | duplicated(count_df2$ENSG, fromLast = TRUE)),] 
rownames(unique_df)=unique_df$ENSG
unique_d2=unique_df %>% dplyr::select(-c("gene", "ENSG", "GENE", "Expression"))

logCPM  = apply(unique_d2, 2, function(x) {log(((x/sum(x)) * 10000 )+1)}) 


dir.create(out_f, recursive=T)
write.table(data.frame(gene=rownames(unique_d2), unique_d2), paste0(out_f, "/low_gene_omitted_count.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

# step2 obtain the intersect genes between TF-GRN matrix and scRNA-seq 

In [ ]:
# ---- Set the following paths to match your environment ----
TF-GRN_path  = "/path/to/TF_GRN_matrix"            # directory for TF_GRN_matrix downloaded from Zenodo
# -----------------------------------------------------------

TF_df=read.table(paste0(TF-GRN_path, "/TF_GRN_matrix.txt", header=1,row.names=1)
TF_df2=TF_df %>% separate(gene_name, c("gene", "ver"), sep="\\.") %>% dplyr::select(-c("ver"))
unique_df=TF_df2[!(duplicated(TF_df2$gene) | duplicated(TF_df2$gene, fromLast = TRUE)),]
rownames(unique_df)=unique_df$gene
unique_df2=unique_df %>% dplyr::select(-c("gene"))
low_exp_omit_path=paste0(out_f, "/low_gene_omitted_count.txt")
out_f3=paste0(out_f, "/gene_filter")
dir.create(out_f3)
sccount_df=read.table(low_exp_omit_path, header=1, row.names=1)
intersect_genes=intersect(rownames(sccount_df), rownames(unique_df2))

##save Gene-TF binding site association matrix
intersect_TF_df=unique_df2[intersect_genes, ]
    
write.table(data.frame(gene=rownames(intersect_TF_df), intersect_TF_df), paste0(out_f3, "/TF_matrix.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

##save RNA-seq logCPM
intersect_sc_df=logCPM[intersect_genes, ]

write.table(data.frame(gene=rownames(intersect_sc_df), intersect_sc_df), paste0(out_f3, "/transcriptome_matrix.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

